# sum-back-expand-broadcast — faded example 3: Sum Backward — complete the dispatch wrapper

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sum-back-expand-broadcast`. Running the beacon reports progress on the `Backprop: sum_back via expand_broadcast` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: sum_back via expand_broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sum-back-expand-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sum-back-expand-broadcast"
DD_SUBTOPIC = "Backprop: sum_back via expand_broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A clean backward implementation for `sum` reads `keepdim` from the stored recipe arguments and dispatches to the right branch: expand directly if `keepdim=True`, or unsqueeze first if `keepdim=False`. Both branches finish with `.expand(x.shape)` because every element of the input contributed to exactly one output element with derivative 1.

## Faded exercise 3

The wrapper below extracts `dim` and `keepdim` from a `kwargs` dict (simulating a recipe). The unsqueeze step is present but the **expand call is missing**. Fill in the blank so the function returns a tensor with the same shape as `x`.

```python
def sum_back_dispatch(grad_out, x, kwargs):
    dim = kwargs['dim']
    keepdim = kwargs.get('keepdim', False)
    g = grad_out if keepdim else grad_out.unsqueeze(dim)
    return None  # TODO: broadcast g to x.shape
```

**Fill in:** Return `g` expanded to `x.shape` using `.expand()`.

In [ ]:
import torch as t

def sum_back_dispatch(grad_out: t.Tensor, x: t.Tensor, kwargs: dict) -> t.Tensor:
    dim = kwargs['dim']
    keepdim = kwargs.get('keepdim', False)
    g = grad_out if keepdim else grad_out.unsqueeze(dim)
    return g.expand(x.shape)

# Test both variants
t.manual_seed(11)
x = t.randn(4, 5)
grad1 = t.ones(4)        # keepdim=False
grad2 = t.ones(4, 1)     # keepdim=True
r1 = sum_back_dispatch(grad1, x, {'dim': 1, 'keepdim': False})
r2 = sum_back_dispatch(grad2, x, {'dim': 1, 'keepdim': True})
print('r1.shape:', r1.shape)  # (4, 5)
print('r2.shape:', r2.shape)  # (4, 5)


def _test():
    import torch as t
    x = t.randn(4, 5)
    # keepdim=False
    grad1 = t.full((4,), 3.0)
    r1 = sum_back_dispatch(grad1, x, {'dim': 1, 'keepdim': False})
    assert r1.shape == x.shape, f'{r1.shape}'
    assert t.allclose(r1, grad1.unsqueeze(1).expand(x.shape))
    # keepdim=True
    grad2 = t.full((4, 1), 3.0)
    r2 = sum_back_dispatch(grad2, x, {'dim': 1, 'keepdim': True})
    assert r2.shape == x.shape
    assert t.allclose(r2, grad2.expand(x.shape))


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

def sum_back_dispatch(grad_out: t.Tensor, x: t.Tensor, kwargs: dict) -> t.Tensor:
    dim = kwargs['dim']
    keepdim = kwargs.get('keepdim', False)
    g = grad_out if keepdim else grad_out.unsqueeze(dim)
    return g.expand(x.shape)

# Test both variants
t.manual_seed(11)
x = t.randn(4, 5)
grad1 = t.ones(4)        # keepdim=False
grad2 = t.ones(4, 1)     # keepdim=True
r1 = sum_back_dispatch(grad1, x, {'dim': 1, 'keepdim': False})
r2 = sum_back_dispatch(grad2, x, {'dim': 1, 'keepdim': True})
print('r1.shape:', r1.shape)  # (4, 5)
print('r2.shape:', r2.shape)  # (4, 5)
```
</details>